## Constructed-data method demonstration: CONSTRUCTED descriptive financial-data analysis

All newly computed values are **CONSTRUCTED method demonstrations**, never market observations or historical verification. Historical tables are labelled separately as replays. French raw, extracted, and processed files are not bundled, and the required rights and permitted uses have not been established; this Notebook does not analyze French data.

Original explanatory prose below describes the course method. The reader-mode code and labels define which portions execute here. No P6 retraining or Ch39 historical row reconstruction.


# Project 1：金融数据分析

本 Notebook 从冻结数据文件开始，依次核验来源与哈希、确认月频和小数单位、按月份对齐 RF，再计算月度收益、财富、风险与固定分段。这个顺序的作用是先保证“每个数字对应哪份数据、哪段时间、哪个单位”，然后才解释指标。

研究对象是 French 发布的非耐用品（NoDur）、高科技（HiTec）和能源（Enrgy）价值加权行业组合月收益。2010–2019 是主窗口；固定的 2010–2014、2015–2019 与 2020–2024 用来观察描述统计对历史区间的敏感度。所有窗口均为事后固定教学样本。

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("project01_financial_data_analysis_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
PROJECT = ROOT / "projects/project01"
sys.path.insert(0, str(PROJECT))
from project01_analysis import load_config, make_project_figure, summarize_all_windows
from quant_course.performance import drawdown_path, wealth_index


## 1. 冻结快照：先核验再分析

`load_project_data` 会先比对本项目清单与课程来源清单，再由共享读取器对实际读取的处理后 CSV 做 SHA-256 校验。数据清单还保留下载 ZIP 与解压 CSV 的哈希，便于追溯“原始下载 → 解压文件 → 处理快照”的转换链。

这里读入的是已经按来源说明清洗过的月度组合收益文件。哈希确保本次分析使用的字节与清单相符；它本身不能证明历史数据当时已经发布。

In [ ]:
config = load_config(PROJECT / "configs/analysis.json")
# CONSTRUCTED: synthetic monthly returns; labels are method examples, not French observations.
rng = np.random.default_rng(101)
index = pd.period_range("2010-01", "2024-12", freq="M", name="month")
data = pd.DataFrame(rng.normal(0.004, 0.035, (len(index), 3)), index=index,
                    columns=[x["column"] for x in config["industries"]])
data["RF"] = 0.001
print("CONSTRUCTED panel:", len(data), "continuous months; French branch SKIPPED: rights unknown")
assert not data.isna().any().any()
data.to_csv(RUN_DIR / "constructed_monthly_returns.csv")


## 2. 时间与单位：为什么要精确对齐

`month` 是该月收益区间的标签，不是月内公告或成交时间。处理文件已把来源百分数除以 100，数值 `0.01` 表示该月收益 1%；再次除以 100 会把收益缩小 100 倍。

RF 是月度无风险收益，因此只与相同月份的行业收益配对。不能拿一个年利率直接减每个月收益，也不应通过填充来制造缺失观测。共享读取器拒绝断月和非有限值，两个输入的 PeriodIndex 必须完全一致。历史逐月发布 / 入库时点、信号和成交均不在这些文件中。

In [ ]:
main = data.loc["2010-01":"2019-12"].copy()
expected_main = pd.period_range("2010-01", "2019-12", freq="M", name="month")
assert main.index.equals(expected_main) and len(main) == 120

display(main.head())
print("主窗口：", main.index[0], "到", main.index[-1], "；", len(main), "个连续月度区间")
print("RF 同窗口月均（百分比 / 月）:", f"{main['RF'].mean() * 100:.4f}%")

## 3. 简单收益、对数收益与财富：先说明计算目的

简单收益（Simple Return）`R_t` 是输入中的月度组合收益。账户没有外部入金时，财富按 `W_t = W_{t-1}(1+R_t)` 更新。乘法来自每月百分比都作用于当时财富；简单收益相加会忽略这个变化的分母。

对数收益（Log Return）`g_t = ln(1+R_t)` 把同一路径的乘积变成可加的和：`Σg_t = ln(W_T/W_0)`。它帮助解释跨时间的复利，但不能直接按权重组合不同资产；本项目仍用每月简单收益构造各自的财富指数。

In [ ]:
# 构造的两期例子只用于手算复利，不作为市场观测。
toy_returns = np.array([0.50, -0.50])
toy_wealth = wealth_index(toy_returns, initial_capital=100.0)
toy_drawdown = drawdown_path(toy_wealth)
print("构造收益:", toy_returns)
print("财富路径:", toy_wealth)
print("路径回撤:", toy_drawdown)
print("算术平均:", toy_returns.mean())
print("平均对数收益:", np.log1p(toy_returns).mean())
np.testing.assert_allclose(toy_wealth, [100.0, 150.0, 75.0])
np.testing.assert_allclose(toy_drawdown, [0.0, 0.0, -0.5])

在实际窗口中，先把来源提供的简单收益变为月度对数收益，再比较两个均值。算术平均主要描述单月收益的平均幅度；平均对数收益对应这条实际收益路径的平均复合增长。对数收益在组合间不能直接按权重相加，所以报告中的财富路径始终调用共享 `wealth_index`，不从对数均值造一条价格曲线。

In [ ]:
mean_returns = pd.DataFrame({
    "industry": [item["label_zh"] for item in config["industries"]],
    "mean_simple_return_pct_per_month": [main[item["column"]].mean() * 100 for item in config["industries"]],
    "mean_log_return_pct_per_month": [np.log1p(main[item["column"]]).mean() * 100 for item in config["industries"]],
})
display(mean_returns.round(4))

## 4. 波动率、同月 RF 夏普与回撤

波动率（Volatility）是月度收益样本标准差。乘 `√12` 得到常见的年化读数，方便在年度尺度上描述离散程度；这是一项约定，不是序列相关存在时年度复利收益标准差的精确等式。

夏普比率（Sharpe Ratio）对每个月先算 `R_t − RF_t`：分子比较当月资产与无风险替代收益，分母衡量超额收益的样本离散程度。随后再乘 `√12`。它是历史窗口描述比率，不是策略显著性或阿尔法（Alpha）检验。

回撤（Drawdown）将当期财富除以截至当期的历史最高财富再减 1。它回答“距历史峰值跌了多少”，因此可与期末累计收益不同。最大回撤按正数展示时，表示峰值至谷底损失比例。

年复合增长率（Compound Annual Growth Rate, CAGR）把窗口初末财富比换算成年度复合率：`(W_end / W_start) ** (12 / n_months) - 1`。它提供统一年度尺度的增长摘要，但会隐藏中间路径，所以与波动率、回撤一起阅读。

In [ ]:
results = summarize_all_windows(data, config)
primary = results.loc[results["window_id"] == config["main_window"]].copy()
main_table = primary[[
    "asset_label_zh", "final_wealth_index", "cumulative_simple_return",
    "cagr_month_period_convention", "annualized_volatility_sqrt12_convention",
    "annualized_sharpe_sqrt12_convention", "maximum_drawdown_loss_fraction",
    "maximum_drawdown_peak_month", "maximum_drawdown_trough_month",
]].rename(columns={
    "asset_label_zh": "行业", "final_wealth_index": "期末财富指数",
    "cumulative_simple_return": "累计收益小数", "cagr_month_period_convention": "CAGR 小数",
    "annualized_volatility_sqrt12_convention": "年化波动率小数",
    "annualized_sharpe_sqrt12_convention": "Sharpe",
    "maximum_drawdown_loss_fraction": "最大回撤损失小数",
    "maximum_drawdown_peak_month": "峰值月", "maximum_drawdown_trough_month": "谷底月",
})
display(main_table.round(4))

### 如何阅读主窗口结果

2010–2019 是 120 个连续月。财富指数从 1 开始，因而期末财富指数 3.14 代表这条收益路径的期末相对财富约为起点的 3.14 倍；它不是一只证券的价格。年复合增长率把初末财富换成十年复合率，不能显示期间先跌后涨的路径。

波动率与最大回撤回答不同风险问题：前者概括每月收益分散，后者量化从此前财富峰值到谷底的最大跌幅。同月 RF 夏普还加入了无风险机会成本，但不意味着组合可以按这个比率交易。先看图中路径，再把这些指标联系起来，比单独按一个数排序更有信息。

## 5. 固定分段：检查描述结论的时间依赖

三个 60 个月区间回答：同一行业指标排序在前后固定历史段是否相同？这种切片只能揭示窗口敏感性；这些窗口都来自当前修订快照，也不是可交易的留出期。

每个窗口都单独以开始前财富 1 归一化；所以分段最大回撤只追踪本段内的峰值，不延续上一段的历史财富峰值。

In [ ]:
segments = results.loc[results["window_id"].isin(["2010_2014", "2015_2019", "2020_2024"]), [
    "window_id", "asset_label_zh", "cagr_month_period_convention",
    "annualized_volatility_sqrt12_convention", "annualized_sharpe_sqrt12_convention",
    "maximum_drawdown_loss_fraction",
]].copy()
segments["CAGR_pct"] = segments.pop("cagr_month_period_convention") * 100
segments["annualized_volatility_pct"] = segments.pop("annualized_volatility_sqrt12_convention") * 100
segments["maximum_drawdown_pct"] = segments.pop("maximum_drawdown_loss_fraction") * 100
display(segments.round(3))

上方是 seed101、180个月的 **CONSTRUCTED 方法演示表**，不是旧 French 市场报告。当前构造能源在2015–2019的 CAGR 约为 +8.008%、Sharpe 约为 +0.623；2020–2024则约为 −0.262% 和 −0.073。正负与上表一致，仅说明这组固定构造输入的分段表现。不能据此选择行业、声称市场阿尔法，或把旧市场报告的另一组区间解释套到这张表。

## 6. 读图并检查轴含义

下图覆盖 2010–2024，灰色部分是主窗口后的扩展段。左上纵轴是财富指数；左下纵轴是相对历史峰值的财富比例差；右上纵轴是滚动 12 个月月度样本标准差乘 `√12` 的年化约定值；右下纵轴是月度简单收益小数，横轴是行业。所有时间横轴把收益区间标签画在月末。

In [ ]:
figure_path = make_project_figure(
    data, RUN_DIR / "project01_dashboard_CONSTRUCTED.png", config
)
print(figure_path)
display(Image(filename=str(figure_path)))

## 7. 解释边界

原 P1 历史市场报告采用已发布的月度价值加权行业收益及同月 RF；其数据版本、结果和许可边界保留在项目报告中。它不含逐股 OHLCV、当时历史版本、精确发布时刻、信号、成交、融资与成本，历史收益或 Sharpe 不是策略表现。

本读者 Notebook 使用 seed101 的180个月构造面板，连同小手算一起演示描述统计、分段表和图的方法；French 实证分支跳过。当前构造表不替代、不复现原市场报告。历史报告与原练习答案分别见 `research_report/report.md`、`exercises.md` 和 `solutions.md`；阅读时须与当前构造输出区分。